# Pretraining

Напомним, как выглядит пайплайн обучения LLM моделей. Он состоит из двух этапов:
1. **Pretraining**<br>self-supervised обучение на терабайтах текста. Цель — научить модель *общему* представлению языка. Дорого (тысячи GPU-месяцев), делается редко, даёт *base-модель*<br><br>
2. **Post-training**<br>SFT (supervised fine-tuning на инструкциях),<br>
выравнивание предпочтений (RLHF / DPO / GRPO и т.п.).

Ключевая идея: почти всё знание модели закладывается на этапе предобучении. Post-training в основном *вытаскивает* и 
*причёсывает* уже усвоенные способности, а не учит новым фактам. Поэтому качество и состав предобучающих данных критично

---

## Как происходит обучение

Предобучение происходит в парадигме [self-supervised learning](https://en.wikipedia.org/wiki/Self-supervised_learning): обучающие примеры генерируются из самого текста, ручная разметка не нужна. Это позволяет условно "бесконечно" масштабироваться — данных в интернете очень много. 

Исторически закрепились два типа моделей

### Авторегрессионная языковая модель (GPT)

Мы требуем чтобы модель предсказывала *следующий токен*, зная все предыдущие токена документа. Это так называемая *causal* модель: делая предсказание для токена $t$ модели доступны токены $x_1, x_2, ... x_{t-1}$. Иногда называют также однонаправленной.

Функция потерь на таких предсказаниях - это обратный логарифм вероятности правильного токена. То есть обычная кросс-энтропия (negative log-likelihood), знакомая по логистической регрессии. 
$$
\mathcal{L}_{\text{LM}} = -\sum_{t=1}^{T} \log p_\theta(x_t \mid x_1, \dots, x_{t-1})
$$

Суммирование идет по всем токенам $1..T$ документа. 

Если от этой функции потерь взять экспоненту, то получаем классическую метрику качества, принятую в языковых моделях — перплексию (для подробностей см главу про классические методы):
$$
\text{PPL} = \exp\!\left(\tfrac{1}{T}\sum_t -\log p_\theta(x_t \mid x_{<t})\right)
$$

Ее грубая интерепретация: «сколько с среднем вариантов продолжнения модель рассматривает при генерации следующего токена». Меньше = лучше

__Почему это работает__<br>
Чтобы хорошо предсказывать следующий токен в произвольном тексте, модели приходится неявно выучить грамматику, факты, причинно-следственные связи, стиль, зачатки рассуждений. Предсказание следующего токена — это, по сути, **сжатие** текста, а хорошее сжатие требует модели мира. Эта установка («prediction = compression = understanding») — идеологический фундамент всей GPT-парадигмы.

### Маскированная языковая модель (BERT)
Задача моделей типа BERT - не продолжение текста слева направо, а восстанавление замаскированных токенов. При этом модель видит весь контекст с обеих сторон (это так называемая *bidirectional* модель)

Две классические цели:

- **MLM (Masked Language Modeling).**<br>Случайно выбираются ~15% токенов; из них (правило 80/10/10): 80% заменяются на `[MASK]`, 10% — на случайный токен, 10% оставляются без изменений. Модель предсказывает оригинал. Трюк 80/10/10 нужен, чтобы модель не «расслаблялась», полагаясь только на наличие `[MASK]` (которого на инференсе не будет).
- **NSP (Next Sentence Prediction).**<br>Подаются две последовательности `[CLS] A [SEP] B [SEP]`; модель решает, идёт ли B сразу за A. Позже (RoBERTa) показали, что NSP скорее вредит, и от неё отказались.

GPT оптимизирует *генеративную* задачу (хорош для порождения текста), BERT — *представленческую* (хорош для понимания, классификации, эмбеддингов). 

Поскольку чат-сценарии требуют генерации, с 2020-х доминирует авторегрессионная парадигма; энкодеры BERT-типа живут в нишах (поиск, ранжирование, эмбеддинги)

**Teacher forcing** — приём обучения авторегрессионных моделей: на каждой позиции в качестве «предыдущих токенов» подаётся истинный контекст из обучающего текста, а не то, что нагенерировала бы сама модель.

Что дает:

- **Параллелизм.** Благодаря teacher forcing все позиции последовательности обучаются *одновременно* за один проход: causal-маска в self-attention гарантирует, что позиция $t$ не «подглядывает» в будущее, а лоссы по всем $t$ считаются разом. Без этого пришлось бы прогонять модель токен за токеном — это убило бы эффективность обучения трансформеров.
- **Стабильность.** Модель учится на корректных префиксах, а не накапливает собственные ошибки во время обучения.

Обратная сторона — **exposure bias**: на обучении модель всегда видела *идеальный* контекст, а на инференсе генерирует **свободно** (feeding собственных предсказаний обратно на вход), поэтому одна ошибка может «потянуть» за собой следующие, уводя в распределение, которого на обучении не было. На практике для больших LLM exposure bias оказался куда менее болезненным, чем опасались, но концептуально его важно понимать: **режим обучения (teacher forcing) и режим генерации (free-running) — это разные режимы**.

---

## 3. Что лежит в типовом батче

Батч для предобучения — это, как правило, тензор формы `[batch_size, sequence_length]` из целочисленных ID токенов плюс служебные тензоры. Различия между GPT- и BERT-стилем показательны.

### 3.1. Батч для GPT (авторегрессия)

- **Входы (`input_ids`)** — упакованные последовательности токенов фиксированной длины (например, 2k–8k, а в современных моделях вплоть до 128k+).
- **Метки (`labels`)** — те же токены, **сдвинутые на одну позицию** (next-token). Часто это делается прямо внутри лосса: `labels[t] = input_ids[t+1]`.
- **Causal attention mask** — нижнетреугольная, запрещает смотреть в будущее.
- **Document packing.** Чтобы не терять вычисления на паддинге, короткие документы **склеивают** в одну длинную последовательность, разделяя специальным токеном (например, `<|endoftext|>` / EOS). Иногда внутри пакета маской запрещают «перетекание» внимания между разными документами (document-aware masking), иногда нет — это инженерный выбор.

Концептуально: **один батч GPT = кусок текста + его же сдвиг как цель**. Разметки нет вообще.

### 3.2. Батч для BERT (MLM)

- **Входы** — последовательности с уже **внесёнными `[MASK]`** (по правилу 80/10/10) и спецтокенами `[CLS]`, `[SEP]`.
- **Метки MLM** — оригинальные токены, но **только на замаскированных позициях** (на остальных лосс не считается, обычно метка `-100`).
- **Segment embeddings** — индикатор «предложение A / предложение B» для пар.
- **Метка NSP** (в оригинальном BERT) — бинарная: «B действительно следует за A» или нет.

Концептуально: **один батч BERT = испорченный текст + задание восстановить выбитые куски** (и, опционально, угадать связность пар).

В GPT лосс считается по *всем* позициям (каждый токен обучает модель), а в BERT — только по ~15% замаскированных. Это одна из причин, почему MLM при равном числе токенов «менее эффективен по сигналу», и почему авторегрессия так хорошо масштабируется

---

## Процесс обучения

### Настройки

- **Оптимизатор**<br>почти всегда **Adam / AdamW** (AdamW = Adam с корректным decoupled weight decay). Настройка $\beta_2$ и $\epsilon$ заметно влияет на стабильность на больших масштабах<br><br>
- **Очень большие батчи**<br>десятки тысяч–миллионы токенов на шаг (за счёт data/tensor/pipeline-параллелизма и градиентной аккумуляции). Большой батч → более «гладкий» градиент → можно держать выше LR<br><br>
- **Mixed precision**<br>обучение в `bf16`/`fp16` с мастер-копией весов в `fp32`; bf16 стал стандартом из-за широкого динамического диапазона<br><br>
- **Регуляризация и стабилизация**<br>weight decay, gradient clipping (обрезка нормы градиента), иногда z-loss на логиты, аккуратная инициализация и нормировки. Цель — пережить **loss spikes** (всплески лосса), которые на больших моделях случаются

### Длительность обучения
Здесь произошёл драматичный сдвиг (см. §9 про scaling laws). Ориентиры по эпохе/масштабу:

- **GPT-3 (2020)** — ~300 млрд токенов.
- **Chinchilla (2022)** — 70B параметров на ~1.4 трлн токенов.
- **LLaMA / Llama 2–3 (2023–2024)** — от ~1–2 трлн до **15 трлн** токенов даже для относительно небольших моделей.
- **DeepSeek-V3 (2024)** — порядка **14.8 трлн** токенов.

Обучение почти всегда идёт **меньше одной эпохи** по уникальным данным (каждый токен видим один раз) либо с небольшим числом повторов — это связано с тем, что повторение данных быстро упирается в насыщение (см. data-constrained scaling в §9).

### Learning rate

Почти все расписания — это разогрев (warmup) + спад (decay)

- **Warmup** — линейный рост LR от 0 до пика за первые сотни–тысячи шагов. Нужен, чтобы Adam-статистики «устаканились» и не разнесли свежеинициализированную модель.
- **Inverse square-root** — `lr ∝ 1/√step` после warmup. Расписание из оригинального трансформера («Attention is All You Need»); сейчас встречается реже.
- **Cosine decay** — после пика LR плавно по косинусу падает до небольшой доли пика (например, до 10%) к концу обучения. **Самое распространённое** расписание для LLM (GPT-3, Llama и др.). Минус: нужно заранее знать общее число шагов — кривая «зашита» под конкретную длину.
- **Linear decay** — линейный спад вместо косинуса; близок по качеству.
- **WSD (Warmup-Stable-Decay)** — современная альтернатива: **warmup → длинная фаза с постоянным LR → короткий резкий decay в самом конце**. Главное преимущество — фаза «stable» не привязана к финальной длине, поэтому обучение легко **продолжать** (continual pretraining) и снимать промежуточные чекпойнты; резкий decay в конце даёт скачок качества. Популяризировано в работах вокруг MiniCPM и используется в ряде современных пайплайнов (включая линейку DeepSeek). Идея — отвязать «накопление» от «дошлифовки».

> **Интуиция.** Decay в конце — это «остывание»: модель из режима быстрого поиска переходит в режим тонкой подгонки. Резкий финальный спад (WSD) часто даёт заметный прирост на бенчмарках за счёт того, что последние шаги делаются на «лучших» (например, более качественных/доменных) данных при малом LR.

---

## 5. Обучающие датасеты и их эволюция

### 5.1. Как менялись данные во времени

Можно выделить несколько эпох:

**Эпоха «чистых» корпусов (2018–2019).** BERT и GPT-1/2 учились на **Wikipedia + BookCorpus** (и WebText у GPT-2 — страницы по исходящим ссылкам с Reddit с минимальным рейтингом). Объёмы — единицы–десятки млрд токенов. Идея: брать относительно «причёсанный» текст.

**Эпоха Common Crawl и фильтрации (2019–2021).**
- **C4** (Colossal Clean Crawled Corpus, для T5, 2019) — ~160–170 млрд токенов: один снапшот Common Crawl, прогнанный через жёсткие эвристики (выкинуть страницы без терминальной пунктуации, короткие, со списком «грязных» слов; дедуп трёхпредложенных спанов). Показал: **из сырого веба можно вычистить пригодный корпус**.
- **The Pile** (2020) — ~340 млрд токенов, **композитный**: 22 источника (книги, статьи arXiv, код GitHub, PubMed, StackExchange и т.д.). Идея: *разнообразие доменов* важно само по себе.
- **GPT-3 mix** — взвешенная смесь: отфильтрованный Common Crawl + WebText2 + книги + Wikipedia, причём более качественные источники **переразвешивались** (видны чаще).

**Эпоха «веб может всё» и масштаба (2022–2023).**
- **RefinedWeb** (Falcon, 2023) — показал, что **аккуратно отфильтрованного веба** (без курируемых источников) достаточно, чтобы догнать/перегнать модели на «чистых» миксах. Извлечение из **WARC** через `trafilatura` (а не из WET, как C4), агрессивный дедуп. Публично выложено ~600 млрд из многотриллионного корпуса.
- **SlimPajama** (~627 млрд) и **RedPajama** — открытые воспроизведения миксов LLaMA-типа с дополнительной чисткой и дедупом.

**Эпоха открытых триллионов и модельной фильтрации (2024+).**
- **Dolma** (AllenAI, ~3 трлн) — открытый корпус с полностью задокументированным пайплайном (fastText для языка, эвристики MassiveText/C4 для качества, дедуп на уровнях URL/документ/абзац, фильтрация токсичности).
- **RedPajama-v2** (~30 трлн) — гигантский набор Common Crawl с **аннотациями** (метками качества) без жёсткой фильтрации: фильтруй сам под задачу.
- **FineWeb** (HuggingFace, **15 трлн**) — 96 снапшотов Common Crawl, **поснапшотный** (а не глобальный) MinHash-дедуп + кастомные фильтры. Открыто и воспроизводимо.
- **FineWeb-Edu** (~1.3 трлн) — **модель-фильтрация по «образовательности»**: классификатор обучен на синтетических метках от Llama-3-70B, оставляет только «учебно-полезный» контент. Даёт то же качество на MMLU/ARC при ~10× меньшем объёме данных. Та же идея ранее использовалась (непублично) в Llama 3 и Phi-3.
- **Доменные мега-корпуса**: **The Stack v2** (~900 млрд токенов кода), доменные математические корпуса и т.д.
- **Синтетические данные**: Phi-линейка («textbooks are all you need»), Nemotron-CC (порядка трети — синтетика) — генерация обучающих текстов другой LLM.

### 5.2. Из чего состоит современный пайплайн обработки данных

Сборка корпуса — это конвейер примерно из таких стадий (порядок и состав варьируются):

1. **Извлечение текста** из HTML/WARC (важен выбор экстрактора: WET vs `trafilatura` по WARC заметно влияет на качество).
2. **Определение языка** (обычно fastText) и отбор нужных языков.
3. **Фильтрация качества**: эвристики (длина, доля пунктуации, доля букв, «спам-словари») **+ модельные классификаторы** качества/образовательности.
4. **Дедупликация**: точная и near-dup (MinHash/LSH) на уровнях URL / документ / абзац. Снимает «горы» дубликатов, которые иначе раздувают и портят корпус.
5. **Декантаминация (decontamination)**: удаление текстов, пересекающихся с тестовыми бенчмарками (например, по совпадению n-грамм), чтобы оценка не была «протекшей».
6. **Удаление PII / токсичности**, фильтры безопасности.
7. **Data mixture / domain weights**: финальное **взвешивание доменов** (сколько кода, сколько вики, сколько веба, сколько математики). Это отдельный важный рычаг качества.

**Главные тренды эволюции данных:** (а) рост масштаба от млрд к десяткам трлн токенов; (б) переход от ручной курации к **автоматической модельной фильтрации**; (в) рост роли **дедупликации** и **декантаминации**; (г) появление **синтетических данных** как полноценного источника; (д) понимание, что **качество и состав смеси важнее «голого» объёма** (см. §9, data pruning).

---

## Итеративная сборка

Два кейса DeepSeek. 

Данные для __DeepSeekMath__ собирались итеративно. Цель — извлечь из Common Crawl большой *математический* корпус размером в ~120 млрд математических токенов:

1. **Seed (затравка)**<br>Берётся небольшой, но хорошо отфильтрованный корпус математических текстов [OpenWebMath](https://arxiv.org/pdf/2310.06786)
2. Обучается классифкатор "математичности" текста<Br>FastText-классификатор (вектор размерности 256). На 500k положительных примерах из seed и 500k отрицательных (случайные веб-страницы) 
3. **Recall.**<br>Обученный классификатор прогоняем по дедуплицированному датасету URL Common Crawl (40 млрд HTML-страниц) и оставляем топ документов
4. **Расширение доменов** <br>Часть найденного размечается людьми; обнаруживается, что узкая затравка пропускает целые матем-домены (форумы, конкретные сайты). Размеченные примеры расширяют представление о том, «как выглядит математика»
5. **Итерация.** <br>Классификатор заново обучается на расширенном наборе и снова прогоняется по вебу — recall растёт. Несколько таких циклов
6. **Дедуп + декантаминация** <br>(удаление пересечений с GSM8K/MATH и т.п.)

### 6.2. DeepSeek-Coder — структурно-осознанная сборка кода

DeepSeek-Coder - модель для кодинга

Состав датасета для обучения: 87% исходный код / 10% англоязычный код-связанный текст (GitHub Markdown, StackExchange) / 3% китайский текст; Всего порядка 2 трлн токенов. 

Конвейер сборки датасета:
1. Crawling открытых репозиториев с GitHub
2. Примениям правила в духе StarCoder: отсечь слишком длинные/битые/автогенерированные файлы и т.п.
3. Внутри каэжого репозитория парсятся зависимости между файлами (по `import` в Python и т.д.) и файлы переупорядочиваются топологически — так, чтобы зависимость шла раньше зависящего файла
4. Зависимые файлы склеиваются в один обучающий пример, чтобы модель видела кросс-файловый контекст (это критично для реального проектного кода)
5. Дедупликация методом MinHash по каждому склеенному репозиторию
6. Фильтрация (удаление файлов, пересекающихся по n-граммам с датасетами HumanEval/MBPP/MATH/GSM8K)

Дополнительно DeepSeek-Coder учится не только на next-token, но и на **FIM (Fill-In-the-Middle)** — документ режется на префикс/середину/суффикс, и модель учится восстанавливать середину по обоим концам (нужно для автодополнения в IDE).

**Идея кейса:** для кода важна **структура** — обработка не пофайловая, а **репозиторная**, с учётом графа зависимостей. Это пример того, как знание домена встраивается прямо в сборку данных.

Сборка доменных данных - это часто «маленький ML-проект внутри ML-проекта»

---

## 7. Чем отличается предобучение VLM (vision-language моделей)

VLM добавляют к тексту зрение. Принципиально новая проблема — **выравнивание (alignment) модальностей**: изображения и текст нужно привести в общее пространство. Сложились несколько архитектурных идей.

### 7.1. Контрастное предобучение (CLIP-линия)

**CLIP** — два энкодера (картиночный и текстовый); обучение **контрастное**: на батче из пар «картинка–подпись» максимизируется сходство правильных пар и минимизируется для неправильных (InfoNCE-лосс). Здесь *нет генерации*: цель — общее эмбеддинг-пространство. Данные — сотни миллионов–миллиарды пар «изображение–текст» из веба (LAION и т.п.). CLIP-энкодер потом часто используется как «глаз» для генеративных VLM.

### 7.2. «Замороженный энкодер + проектор + LLM» (LLaVA-линия)

Самая распространённая практичная схема:

- Берётся **готовый зрительный энкодер** (например, CLIP ViT), обычно **замороженный**.
- Картинка превращается в набор **визуальных токенов**; небольшой обучаемый **проектор** (MLP) отображает их в эмбеддинг-пространство LLM.
- Визуальные токены **конкатенируются** с текстовыми и подаются в обычную LLM.
- Обучение часто **двухстадийное**: (1) *alignment pretraining* — учим в основном проектор на парах «картинка–описание»; (2) *instruction tuning* на мультимодальных диалогах.

Здесь цель по-прежнему **next-token prediction на текстовой части** — то есть «генеративная голова» осталась той же, что у LLM. Это важная мысль: **большинство современных VLM остаются авторегрессионными по тексту**, просто текстовый контекст дополнен визуальными токенами.

### 7.3. Cross-attention и interleaved-данные (Flamingo-линия)

**Flamingo** держит и LLM, и зрительный энкодер **замороженными**, а связывает их через вставляемые **cross-attention**-слои; обучается на **чередующихся (interleaved)** последовательностях «текст–картинка–текст», что даёт few-shot мультимодальность.

### 7.4. Нативно-мультимодальное предобучение

Новейший тренд — учить модель мультимодальной **с нуля**, смешивая текст, изображения (а иногда аудио/видео) в одном потоке токенов и в одном предобучении, без «пришивания» зрения постфактум.

**Что отличается от чисто текстового предобучения:**
- Нужны **парные/чередующиеся** данные (картинка–текст), а не только текст; источники — веб-альт-тексты, interleaved-документы, OCR, **синтетические подписи** (часто сгенерированные другой моделью).
- Появляется **контрастная** цель (в CLIP-части) наряду с генеративной.
- Возникают свои узлы: разрешение и токенизация изображений, число визуальных токенов на картинку (баланс «детализация vs длина контекста»), выбор «что замораживать».

**Что остаётся тем же:** трансформер, teacher forcing по тексту, кросс-энтропия next-token, расписания LR, scaling-логика. То есть **VLM — это в основном надстройка над той же текстовой парадигмой**, плюс модуль выравнивания модальностей.

---

## 8. Хаки и эффекты на этапе предобучения

### 8.1. Grokking (отложенное обобщение)

**Grokking** (Power et al., 2022) — эффект, замеченный на игрушечных задачах (модульная арифметика и т.п.): модель сначала **запоминает** обучающую выборку (train-accuracy уходит в ~100%, а val-accuracy у пола), и лишь **спустя очень много шагов после переобучения** валидационная точность внезапно подскакивает до near-perfect — происходит «озарение».

Что важно понять:
- Это **фазовый переход** «зубрёжка → понимание»: внутри сети формируется обобщающая структура уже после видимого переобучения.
- Ключевую роль играет **регуляризация** (особенно weight decay): без неё переход к обобщению может не наступать. Это намёк, что обобщение «выгоднее» простому решению при правильном давлении на простоту.
- На масштабе настоящих LLM grokking в чистом виде наблюдают редко, но как **концепция** он важен: «плато по метрике ≠ модель не учится».

### 8.2. Emergent abilities (возникающие способности)

**Emergent abilities** (Wei et al., 2022) — способности, которых **нет у маленьких моделей и которые «включаются» при достижении некоторого масштаба** (многозначная арифметика, некоторые формы in-context learning, прохождение определённых задач). На графике «качество vs масштаб» это выглядит как почти ступенька: около нуля, около нуля — и вдруг резкий рост.

### 8.3. Критика «эмерджентности как миража»

**Schaeffer et al. (2023)** возразили: значительная часть «скачков» — **артефакт выбора метрики**. Если мерить *разрывной/пороговой* метрикой (например, exact-match: всё-или-ничего), кривая выглядит ступенчатой; если взять *гладкую/непрерывную* метрику (например, по-токенную вероятность правильного ответа), та же способность растёт **плавно и предсказуемо**. Вывод: эмерджентность может быть свойством **линейки измерения**, а не самой модели.

> **Как это держать в голове.** Истина где-то посередине: метрики действительно создают часть «ступенек», но и реальные качественные переходы в поведении наблюдаются. Для собеседования полезно уметь назвать *обе* стороны: Wei (есть эмерджентность) и Schaeffer (метрический мираж).

### 8.4. Прочие эффекты и инженерные «хаки»

- **Double descent** — немонотонная зависимость ошибки от размера модели/числа эпох: тест-ошибка падает, растёт у «интерполяционного порога», затем снова падает. Ломает классическую U-образную интуицию bias–variance.
- **Loss spikes (всплески лосса)** — внезапные скачки лосса на больших моделях. Лечат снижением LR, **пропуском «плохих» батчей**, перезапуском с предыдущего чекпойнта, нормировкой эмбеддингов/логитов, аккуратной настройкой $\beta_2$ у Adam.
- **Memorization** — модель буквально запоминает части обучающих данных (важно для приватности и для честности оценки → отсюда декантаминация).
- **Финальный decay как «хак качества»** — резкое снижение LR в конце (особенно в WSD) и подмешивание самых качественных/доменных данных на последних шагах дают ощутимый прирост на бенчмарках.

---

## 9. Scaling Laws для предобучения

Scaling laws — это **эмпирические степенные закономерности**: как падает лосс с ростом числа параметров $N$, объёма данных $D$ и вычислений $C$. Их ценность — **предсказуемость**: можно по маленьким прогонам экстраполировать, что даст большой, и заранее распределить бюджет.

### 9.1. Kaplan et al. (2020) — «модель важнее»

Первая систематическая работа (OpenAI). Главные тезисы:
- Лосс падает по **степенному закону** относительно $N$, $D$ и $C$ — гладко и предсказуемо на много порядков.
- При фиксированном бюджете вычислений выгоднее **сильно увеличивать модель** и относительно умеренно — данные. В их оценках оптимальный размер модели рос как $N_{\text{opt}} \propto C^{\,0.73}$.

Эта логика легитимизировала гонку «больше параметров» (кульминация — GPT-3, 175B на «всего» 300B токенов).

### 9.2. Chinchilla / Hoffmann et al. (2022) — «данные недооценили»

DeepMind переобучили **400+ моделей** при контролируемых бюджетах и предложили параметрическую форму лосса:

$$
L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}
$$

где $E$ — неустранимая «энтропия» данных, а два члена — вклад ограниченной ёмкости модели и ограниченных данных. Минимизация $L$ при фиксированном $C \approx 6ND$ дала вывод: **модель и данные надо растить примерно одинаково**, $N_{\text{opt}} \propto C^{0.5}$, $D_{\text{opt}} \propto C^{0.5}$.

Знаменитое практическое правило — **≈ 20 токенов на параметр** (отношение $D/N \approx 20$). Демонстрация: **Chinchilla-70B на 1.4 трлн токенов обошла Gopher-280B на 300 млрд токенов при ~4× меньших вычислениях**. Вывод-шок: **прежние большие модели были недообучены** (слишком много параметров на слишком мало данных).

### 9.3. Примирение Kaplan и Chinchilla

Расхождение показателей ($0.73$ против $0.5$) оказалось во многом **методологическим** (Pearce et al., 2024; Porian et al., 2024): Kaplan считал **не-эмбеддинговые** параметры и работал на меньших масштабах; при корректном учёте эмбеддингов, FLOP-ов «головы», длины warmup и настроек оптимизатора результаты сходятся к Chinchilla. Независимая репликация (Besiroglu et al., 2024) подтвердила оптимум около **20–25 токенов на параметр**. Сейчас сообщество в целом принимает **Chinchilla-режим** как базовый для compute-optimal.

### 9.4. Data-constrained scaling — когда данные кончаются

Muennighoff et al. (2023) исследовали обучение при **ограниченном** объёме уникальных данных. Главный результат: **повторять данные можно, но недолго** — примерно **до ~4 эпох** повтор почти эквивалентен свежим токенам, после чего отдача быстро падает, и добавлять параметры/повторы становится почти бесполезно. Это формализует «**data wall**» — упор в исчерпание качественных данных — и мотивирует синтетику и более жёсткую фильтрацию.

### 9.5. Pruning-based scaling — «качество бьёт степенной закон»

Sorscher et al. (2022), *«Beyond neural scaling laws: beating power law scaling via data pruning»*: если есть **хорошая метрика отбора примеров** (какие выбрасывать первыми), то ошибку можно ронять **быстрее степенного закона — вплоть до экспоненциального** убывания по размеру (оставленного) датасета. Иначе говоря, **не всякий токен одинаково полезен**: грамотный отбор данных принципиально меняет кривую масштабирования. Это теоретическая опора под всю волну модель-фильтрации (FineWeb-Edu, Phi и т.п.): **состав и качество данных — рычаг не меньший, чем объём**.

### 9.6. Inference-aware scaling — почему модели сегодня «переобучают»

Sardana et al. (2023), *«Beyond Chinchilla-Optimal»*: Chinchilla оптимизирует только **стоимость обучения**, но в реальности модель потом обслуживает **миллиарды запросов**. Если учесть **стоимость инференса**, оптимально брать модель **меньше**, но обучать её на **гораздо большем** числе токенов — маленькую модель дешевле и обучать «с запасом», и потом эксплуатировать.

Именно поэтому современные релизы **сознательно уходят далеко за «20 токенов/параметр»**: например, Llama-3-8B обучали на ~15 трлн токенов (это **~1875 токенов на параметр** — на два порядка «переобучение» по Chinchilla). С точки зрения compute-optimal это «расточительно», с точки зрения **inference-optimal** — рационально.

### 9.7. Что ещё бывает (для полноты)

- **Scaling laws для трансфера и дообучения** — как эффект предобучения переносится на downstream при дообучении.
- **Scaling laws для MoE (Mixture-of-Experts)** — отдельные зависимости для разреженных моделей (активных vs всего параметров).
- **Мультимодальные scaling laws** — для VLM/контрастного обучения.
- **Scaling laws для данных и фильтрации** — как качество фильтра сдвигает кривые (продолжение идеи §9.5).

> **Эволюция мысли в одну строку.** *«Растите модель»* (Kaplan) → *«растите и данные тоже, вы недообучили»* (Chinchilla) → *«данные конечны, повтор не спасает»* (data-constrained) → *«качество данных меняет сам закон»* (pruning) → *«считайте ещё и инференс, поэтому переобучайте маленькие модели»* (inference-aware).

---

## 10. Резюме: как развивалась мысль

**Цель обучения.** От конкуренции двух парадигм (генеративная GPT vs представленческая BERT) — к доминированию **авторегрессии** как самой масштабируемой self-supervised задачи. Teacher forcing — то, что делает её обучаемой параллельно.

**Данные.** От «чистых» Wikipedia/BookCorpus → к фильтрованному вебу (C4) → к разнообразным композитным корпусам (The Pile) → к доказательству «веба достаточно» (RefinedWeb) → к открытым триллионам (Dolma, FineWeb) → к **модельной фильтрации и синтетике** (FineWeb-Edu, Phi, Nemotron-CC). Параллельно — рост роли **дедупа, декантаминации и взвешивания смеси**, и появление **итеративных, структурно-осознанных** сборок (DeepSeekMath, DeepSeek-Coder).

**Масштабирование.** От «модель важнее» (Kaplan) к compute-optimal балансу (Chinchilla) и далее к признанию, что **качество данных и стоимость инференса** меняют оптимум сильнее, чем голый объём.

**Эффекты.** Grokking и emergent abilities научили нас осторожности с метриками и с интуицией «плато = застой»; критика «миража» — тому, что часть скачков создаётся самой линейкой измерения.

---

## Приложение A. Ключевые работы (для справки и собеседований)

| Тема | Работа | Год | Идея в одной фразе |
|---|---|---|---|
| Архитектура | Vaswani et al., *Attention Is All You Need* | 2017 | Трансформер; self-attention; inverse-sqrt LR |
| Энкодер-парадигма | Devlin et al., *BERT* | 2018 | MLM + NSP, bidirectional, маскирование 15% |
| Авторегрессия | Radford et al., *GPT-2 / GPT-3* | 2019/2020 | Next-token LM; few-shot; данные-смесь |
| Корпус | Raffel et al., *T5 / C4* | 2019 | Чистка одного снапшота Common Crawl |
| Корпус | Gao et al., *The Pile* | 2020 | Композитный корпус из 22 доменов |
| Scaling | Kaplan et al., *Scaling Laws* | 2020 | Степенные законы; «растите модель» |
| Scaling | Hoffmann et al., *Chinchilla* | 2022 | Compute-optimal; ~20 токенов/параметр |
| Эффект | Power et al., *Grokking* | 2022 | Отложенное обобщение после переобучения |
| Эффект | Wei et al., *Emergent Abilities* | 2022 | Способности «включаются» с масштабом |
| Данные | Sorscher et al., *Beyond Neural Scaling Laws* | 2022 | Хороший отбор данных бьёт степенной закон |
| Корпус | Penedo et al., *RefinedWeb* | 2023 | Отфильтрованного веба достаточно |
| Данные | Muennighoff et al., *Data-Constrained LMs* | 2023 | Повтор данных полезен до ~4 эпох |
| Критика | Schaeffer et al., *Emergence a Mirage?* | 2023 | Скачки — артефакт разрывных метрик |
| Scaling | Sardana et al., *Beyond Chinchilla-Optimal* | 2023 | Учёт инференса → «переобучай» малые модели |
| Корпус | Soldaini et al., *Dolma* | 2024 | Открытый 3T-корпус с задокументированным пайплайном |
| Данные | Penedo et al., *FineWeb / FineWeb-Edu* | 2024 | 15T + модельная фильтрация по «образовательности» |
| Доменные данные | Guo et al., *DeepSeek-Coder* | 2024 | Структурно-осознанная сборка кода (граф зависимостей) |
| Доменные данные | Shao et al., *DeepSeekMath* | 2024 | Итеративная классификатор-управляемая добыча |

## Приложение B. Мини-словарь

- **Self-supervised learning** — обучение без ручной разметки; метки берутся из самих данных.
- **Teacher forcing** — подача истинного префикса вместо собственных предсказаний модели на обучении.
- **Exposure bias** — расхождение между обучением (истинный контекст) и инференсом (свой контекст).
- **Perplexity** — экспонента средней кросс-энтропии; мера «неуверенности» модели.
- **Packing** — склейка коротких документов в длинные последовательности ради эффективности.
- **Warmup / decay** — фазы роста и спада learning rate.
- **WSD** — Warmup-Stable-Decay: расписание с длинной стабильной фазой и резким спадом в конце.
- **Dedup / decontamination** — удаление дубликатов / удаление пересечений с тестами.
- **Data mixture** — пропорции доменов в обучающей смеси.
- **Compute-optimal** — выбор $N$ и $D$, минимизирующий лосс при фиксированных вычислениях.
- **FIM (Fill-In-the-Middle)** — цель: восстановить середину по префиксу и суффиксу (для кода).